<a href="https://colab.research.google.com/github/woraphonp-038-5/Project_LineChatbot_Contraception/blob/main/Chatbot_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### **1.ติดตั้ง Dependencies**

In [ ]:
!pip install -q groq supabase sentence-transformers langchain langchain-community langchain-groq psycopg2-binary;

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


#### **2.ตั้งค่า API Keys**

In [ ]:
import os

# วิธีที่ 1
GROQ_API_KEY       = "---------"
SUPABASE_URL       = "---------"   # เช่น https://xxxx.supabase.co
SUPABASE_KEY       = "---------"
HUGGINGFACE_TOKEN  = "---------"      # ต้องใช้ถ้า BAAI/bge-m3 ต้องการ auth

# วิธีที่ 2 ใช้ Colab Secrets แทน
# from google.colab import userdata
# GROQ_API_KEY      = userdata.get("GROQ_API_KEY")
# SUPABASE_URL      = userdata.get("SUPABASE_URL")
# SUPABASE_KEY      = userdata.get("SUPABASE_KEY")
# HUGGINGFACE_TOKEN = userdata.get("HUGGINGFACE_TOKEN")

os.environ["GROQ_API_KEY"]        = GROQ_API_KEY
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACE_TOKEN

#### **3.Import Libraries**

In [ ]:
import json
from typing import List, Dict, Optional
from groq import Groq
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer
import psycopg2
import json
import uuid

#### **4.เชื่อมต่อ Supabase และโหลด Embedding Model**

In [ ]:
print("🔗 กำลังเชื่อมต่อ Supabase...")
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Supabase เชื่อมต่อสำเร็จ")

print("📦 กำลังโหลด Embedding Model (BAAI/bge-m3)...")
embedding_model = SentenceTransformer("BAAI/bge-m3")
print("✅ Embedding Model โหลดสำเร็จ")

groq_client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq Client พร้อมใช้งาน")

🔗 กำลังเชื่อมต่อ Supabase...
✅ Supabase เชื่อมต่อสำเร็จ
📦 กำลังโหลด Embedding Model (BAAI/bge-m3)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embedding Model โหลดสำเร็จ
✅ Groq Client พร้อมใช้งาน


#### **5.กำหนดค่า Vector Store Tables (ตรงกับ n8n nodes ทุก table)**

In [ ]:
VECTOR_STORES = [
    {
        "name": "WHO MEC 2025 6th edition",
        "table": "WHO_MEC_2025_6th_edition",
        "function": "match_who_mec_2025_6th_edition",
        "top_k": 3,
        "description": (
            "WHO evidence-based guidelines on contraception and family planning, "
            "including MEC and SPR recommendations. Covers contraceptive safety, "
            "eligibility, contraindications, and appropriate use based on health conditions."
        ),
    },
    {
        "name": "WHO SPR 2025 4th edition",
        "table": "WHO_SPR_2025_4th_edition",
        "function": "match_who_spr_2025_4th_edition",
        "top_k": 3,
        "description": (
            "WHO family planning and contraception guidelines including SPR and MEC recommendations. "
            "Covers safe contraceptive use, counseling, informed consent, reproductive health rights, "
            "and clinical best practices."
        ),
    },
    {
        "name": "WHO family planning 2022",
        "table": "WHO_family_planning_2022",
        "function": "match_who_family_planning_2022",
        "top_k": 3,
        "description": (
            "2022 WHO Family Planning Global Handbook for Providers. Covers contraceptive methods, "
            "counseling, STI care, HIV considerations, postabortion care, self-injection (DMPA-SC), "
            "and reproductive health service guidelines."
        ),
    },
    {
        "name": "GuidelinesSTI",
        "table": "GuidelinesSTI",
        "function": "match_guidelinessti",
        "top_k": 3,
        "description": (
            "Thailand STI and HIV epidemiology, prevention, diagnosis, treatment, and clinical care "
            "guidelines. Includes infection trends, risk groups, symptoms, treatment protocols, "
            "and 2024 clinical practice recommendations."
        ),
    },
    {
        "name": "Sexually Transmitted Infections",
        "table": "Sexually_Transmitted_Infections",
        "function": "match_sexually_transmitted_infections",
        "top_k": 3,
        "description": (
            "CDC evidence-based STI treatment guidelines for prevention, diagnosis, counseling, "
            "and clinical management. Covers gonorrhea, chlamydia, syphilis, HPV, herpes, "
            "hepatitis C, PID, Mycoplasma genitalium, screening, and treatment recommendations."
        ),
    },
]

#### **6.System Prompt**

In [ ]:
SYSTEM_PROMPT = """You are a friendly sexual health assistant.
Speak Thai in a warm, supportive, non-judgmental tone like a close friend.

Rules:
- Call yourself "เรา"
- Call the user "แก", "เธอ", or by name naturally
- Use simple, empathetic language
- Keep medical information accurate and evidence-based
- For STI or contraception questions, always use retrieved database context before answering
- Never make up medical information
- If information is unclear or symptoms seem serious, recommend seeing a doctor

Response style:
1. Start with empathy and emotional support
2. Answer using retrieved knowledge only
3. Explain in easy, conversational Thai
4. Encourage safe practices naturally"""

#### **7.ฟังก์ชัน Retrieve จาก Supabase Vector DB**

In [ ]:
def get_embedding(text: str) -> List[float]:
    """สร้าง embedding จาก BAAI/bge-m3"""
    embedding = embedding_model.encode(text, normalize_embeddings=True)
    return embedding.tolist()


def retrieve_from_store(store: dict, query_embedding: List[float]) -> List[str]:
    """
    เรียก RPC function ใน Supabase สำหรับ vector similarity search
    ต้องมี match_* function ใน Supabase ก่อน (ดู NOTE ด้านล่าง)
    """
    try:
        result = supabase.rpc(
            store["function"],
            {
                "query_embedding": query_embedding,
                "match_count": store["top_k"],
            },
        ).execute()

        if result.data:
            # ดึง content จากแต่ละ row (ปรับ field name ให้ตรงกับ schema ของคุณ)
            texts = []
            for row in result.data:
                content = row.get("content") or row.get("text") or row.get("document") or str(row)
                texts.append(content)
            return texts
        return []

    except Exception as e:
        print(f"⚠️  ดึงข้อมูลจาก {store['name']} ไม่ได้: {e}")
        return []


def retrieve_all_context(user_query: str) -> str:
    """
    ดึงข้อมูลจากทุก vector store แล้วรวมเป็น context เดียว
    (เหมือนกับที่ n8n agent ใช้ tools ทั้ง 5 ตัวพร้อมกัน)
    """
    print("🔍 กำลังค้นหาข้อมูลจาก Vector Database...")
    query_embedding = get_embedding(user_query)

    all_contexts = []
    for store in VECTOR_STORES:
        docs = retrieve_from_store(store, query_embedding)
        if docs:
            block = f"### {store['name']}\n" + "\n\n".join(docs)
            all_contexts.append(block)
            print(f"   ✅ {store['name']}: พบ {len(docs)} เอกสาร")
        else:
            print(f"   — {store['name']}: ไม่พบข้อมูลที่เกี่ยวข้อง")

    if not all_contexts:
        return ""

    return "\n\n---\n\n".join(all_contexts)

#### **8.Postgres Memory เชื่อมต่อ n8n_chat_histories**

In [ ]:
# ใส่ค่าจาก Supabase → Settings → Database → Connection string
PG_HOST     = "---------"
PG_PORT     = 65
PG_DATABASE = "postgres"
PG_USER     = "---------"
PG_PASSWORD = "---------"

# สร้าง session_id ใหม่ทุกครั้งที่เปิด Colab
# หรือกำหนดเองเพื่อให้ต่อเนื่องจากครั้งก่อน
SESSION_ID = str(uuid.uuid4())
print(f"📌 Session ID: {SESSION_ID}")

def get_pg_conn():
    return psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        database=PG_DATABASE, user=PG_USER, password=PG_PASSWORD
    )

def save_message(role: str, content: str):
    """บันทึกข้อความลง n8n_chat_histories"""
    conn = get_pg_conn()
    cursor = conn.cursor()
    message_json = json.dumps(
        {"type": role, "content": content},
        ensure_ascii=False
    )
    cursor.execute("""
        INSERT INTO n8n_chat_histories (session_id, message)
        VALUES (%s, %s::jsonb)
    """, (SESSION_ID, message_json))
    conn.commit()
    conn.close()

def load_recent_history(window_size: int = 3) -> list:
    """โหลดประวัติ 3 รอบล่าสุดสำหรับส่งให้ Groq"""
    conn = get_pg_conn()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT message FROM n8n_chat_histories
        WHERE session_id = %s
        ORDER BY id DESC
        LIMIT %s
    """, (SESSION_ID, window_size * 2))
    rows = cursor.fetchall()
    conn.close()

    messages = []
    for row in reversed(rows):
        msg = row[0]
        if isinstance(msg, str):
            msg = json.loads(msg)
        role = "user" if msg.get("type") == "human" else "assistant"
        messages.append({"role": role, "content": msg.get("content", "")})
    return messages

print("✅ Postgres Memory เชื่อมต่อสำเร็จ")

📌 Session ID: f92298e9-4000-43a2-be28-3f5d961ac7d0
✅ Postgres Memory เชื่อมต่อสำเร็จ


#### **9.ฟังก์ชัน chat()**

In [ ]:
def chat(user_message: str) -> str:
    # 1. ดึงบริบทจาก Vector DB
    context = retrieve_all_context(user_message)

    if context:
        user_content = (
            f"ข้อมูลอ้างอิงจากฐานข้อมูล:\n\n{context}\n\n"
            f"---\n\nคำถามของผู้ใช้: {user_message}"
        )
    else:
        user_content = user_message

    # 2. บันทึกคำถามลง Postgres
    save_message("human", user_content)

    # 3. โหลดประวัติ 3 รอบล่าสุดจาก Postgres
    history = load_recent_history(window_size=3)

    # 4. ส่งให้ Groq
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history
    print("🤖 กำลังสร้างคำตอบ...")
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        temperature=0.7,
        max_tokens=1024,
    )
    assistant_reply = response.choices[0].message.content

    # 5. บันทึกคำตอบลง Postgres
    save_message("assistant", assistant_reply)

    return assistant_reply

print("✅ chat() พร้อมใช้งาน")

✅ chat() พร้อมใช้งาน


#### **10.Chat Loop (รันเพื่อใช้งาน Chatbot)**

In [ ]:
def run_chatbot():
    """
    เริ่มต้น chatbot แบบ interactive
    พิมพ์ 'quit' หรือ 'exit' เพื่อออก
    """
    print("🤖 Chatbot พร้อมใช้งาน!")
    print("พิมพ์ 'quit' เพื่อออก")

    while True:
        print()
        user_input = input("คุณ: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ["quit", "exit", "ออก"]:
            print("👋 ลาก่อนนะ ดูแลสุขภาพด้วยนะ!")
            break

        print()
        response = chat(user_input)
        print(f"บอท: {response}")
        print("-" * 60)

#### **Run_chatbot**

In [ ]:
run_chatbot()

🏥 Chatbot พร้อมใช้งาน!
พิมพ์ 'quit' เพื่อออก

คุณ: hi

🔍 กำลังค้นหาข้อมูลจาก Vector Database...
   ✅ WHO MEC 2025 6th edition: พบ 3 เอกสาร
   ✅ WHO SPR 2025 4th edition: พบ 3 เอกสาร
   ✅ WHO family planning 2022: พบ 3 เอกสาร
   ✅ GuidelinesSTI: พบ 3 เอกสาร
   ✅ Sexually Transmitted Infections: พบ 3 เอกสาร
🤖 กำลังสร้างคำตอบ...
บอท: สวัสดีค่ะ 😊  
เราอยู่ตรงนี้เพื่อช่วยเธอทุกเรื่องเลย ไม่ว่าต้องการคำแนะนำด้านสุขภาพทางเพศ, คุมกำเนิด, หรือแม้แต่เรื่องเล็ก ๆ ที่อยากแชร์บ้างก็บอกมาได้เลยนะคะ คุณต้องการพูดคุยเรื่องอะไรเป็นพิเศษหรือเปล่าคะ?
------------------------------------------------------------

คุณ: ต้องการคำแนะนำด้านสุขภาพทางเพศ

🔍 กำลังค้นหาข้อมูลจาก Vector Database...
   ✅ WHO MEC 2025 6th edition: พบ 3 เอกสาร
   ✅ WHO SPR 2025 4th edition: พบ 3 เอกสาร
   ✅ WHO family planning 2022: พบ 3 เอกสาร
   ✅ GuidelinesSTI: พบ 3 เอกสาร
   ✅ Sexually Transmitted Infections: พบ 3 เอกสาร
🤖 กำลังสร้างคำตอบ...
บอท: สวัสดีค่ะ 🌸  
เราเข้าใจว่าการดูแลสุขภาพทางเพศเป็นเรื่องสำคัญมาก และอาจทำให้รู้สึกกังว